# Region ablation, done properly

The first attempt did not actually test the object hypothesis. KMeans with a spatial weight produced
**3.62 connected components per region** — confetti, not objects — so the sharper drops it reported
("77% -> 30% near-zero") were just *masking 10 patches hurts more than masking 1*, which is trivially
true and tells us nothing.

## Three fixes

**1. Contiguous by construction.** Agglomerative clustering with a **grid connectivity constraint**,
so every region is a connected blob by definition rather than by tuning a weight. The notebook
**asserts components == 1.0 before spending a single forward pass.**

**2. A size-matched random control.** For every region of size *m*, also ablate *m* patches chosen
at random. This is what separates "objects matter" from "bigger masks hurt more" — the control the
first version was missing.

**3. A super-additivity test, free from the cache.** We already have per-patch drops. If patches
inside an object are redundant, then removing the **whole** object should hurt *more* than the sum of
removing its patches one at a time:

```
ratio = drop(whole region) / sum( drop(patch i) for i in region )

ratio >> 1  ->  redundancy: patches cover for each other, per-patch ablation understates importance
ratio ~= 1  ->  additive: no redundancy, per-patch ablation was fine all along
```

That single number decides whether ten notebooks of patch-level work were mis-measured.

Plus a K sweep (4 / 8 / 12) so the conclusion does not hinge on one arbitrary region count.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy scikit-learn
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, time, gc
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import wilcoxon
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.feature_extraction.image import grid_to_graph

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import visual_selection as VS

N100  = "/content/drive/MyDrive/wearvqa_n100.pt"
EMB   = "/content/drive/MyDrive/wearvqa_n100_emb.pt"
SINKF = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
OUT   = "/content/drive/MyDrive/wearvqa_regions_v2.pt"
MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
KS, PCA_DIM = (4, 8, 12), 16

for p in (N100, EMB):
    assert os.path.exists(p), f"{p} missing - run colab_n100_scaleup then colab_train_frm"
data  = [d for d in torch.load(N100, weights_only=False) if "gp" in d]
embs  = torch.load(EMB, weights_only=False)
sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()

L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)
CONN = grid_to_graph(G, G)                      # 4-connected 9x9 grid, C order == our indexing
print(f"{N} examples | L_v={L_v} ({G}x{G}) | K sweep {KS}")

## 2. Contiguous regions — and prove it before spending any GPU

In [ ]:
def regions_for(E, k):
    Z = PCA(PCA_DIM, random_state=0).fit_transform(F.normalize(E.float(), dim=-1).numpy())
    return torch.tensor(AgglomerativeClustering(
        n_clusters=k, connectivity=CONN, linkage="ward").fit_predict(Z))

def n_components(mask2d):
    seen = np.zeros_like(mask2d, dtype=bool); n = 0
    for i in range(G):
        for j in range(G):
            if mask2d[i, j] and not seen[i, j]:
                n += 1; st = [(i, j)]; seen[i, j] = True
                while st:
                    a, b = st.pop()
                    for da, db in ((1,0),(-1,0),(0,1),(0,-1)):
                        p, q = a+da, b+db
                        if 0 <= p < G and 0 <= q < G and mask2d[p, q] and not seen[p, q]:
                            seen[p, q] = True; st.append((p, q))
    return n

LAB = {k: [regions_for(e, k) for e in embs] for k in KS}

for k in KS:
    comps, sz = [], []
    for l in LAB[k]:
        for r in range(k):
            m = (l.reshape(G, G) == r).numpy()
            if m.any():
                comps.append(n_components(m)); sz.append(int(m.sum()))
    mc = float(np.mean(comps))
    print(f"K={k:<3} components/region {mc:.3f}   size mean {np.mean(sz):.1f} "
          f"min {np.min(sz)} max {np.max(sz)}")
    assert mc == 1.0, (f"K={k}: regions are not contiguous ({mc:.2f}) - "
                       "the object hypothesis cannot be tested with these")
print("\nall regions contiguous by construction. Safe to spend GPU.")

In [ ]:
from PIL import Image as _I
k = 8
fig, ax = plt.subplots(2, 4, figsize=(15, 7))
for a, i in zip(ax.ravel(), range(8)):
    img = S.load_image(data[i]["img_path"]); W, H = img.size
    seg = _I.fromarray((LAB[k][i].reshape(G, G).numpy() * (255 // k)).astype("uint8")).resize((W, H), _I.NEAREST)
    a.imshow(img); a.imshow(np.array(seg), cmap="tab10", alpha=0.55)
    gp = data[i]["gp"]
    a.scatter([(gp % G + .5)/G*W], [(gp // G + .5)/G*H], marker="x", s=140, c="lime", linewidths=3)
    a.set_title(data[i]["type"][:24], fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()
print("These should read as surfaces and objects. Compare with the confetti from v1.")

## 3. Ablate regions, and size-matched random sets

Sinks are never masked, so a region containing a corner cannot get credit for destabilising the
model. The random control draws the same number of non-sink patches, so any advantage the regions
show is about *grouping*, not about *size*.

In [ ]:
model, processor, device = S._load_smolvlm(MODEL_ID)

def build_inputs(image, question, answer):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt-1:].sum())

rng = np.random.default_rng(0)
ok = (~sinks[:L_v]).nonzero().squeeze(-1).numpy()

if os.path.exists(OUT):
    res = torch.load(OUT, weights_only=False)
    print(f"resuming from {len(res)} examples")
else:
    res = []

t0 = time.time()
for i in range(len(res), N):
    d = data[i]
    inp, n_prompt = build_inputs(S.load_image(d["img_path"]), d["question"], d["answer"])
    ids = inp["input_ids"][0].cpu()
    img_pos = torch.nonzero(ids == S._find_image_token_id(model, processor)).squeeze(-1)
    base = answer_logprob(inp, n_prompt)

    def ablate(sel):
        if len(sel) == 0:
            return 0.0
        am = inp["attention_mask"].clone(); am[0, img_pos[np.asarray(sel)]] = 0
        return base - answer_logprob(inp, n_prompt, am)

    rec = dict(idx=i, base=base)
    for k in KS:
        lab = LAB[k][i]
        rd, rnd, msz = torch.zeros(k), torch.zeros(k), torch.zeros(k)
        for r in range(k):
            sel = ((lab == r) & (~sinks[:L_v])).nonzero().squeeze(-1).numpy()
            msz[r] = len(sel)
            rd[r]  = ablate(sel)
            rnd[r] = ablate(rng.choice(ok, size=min(len(sel), len(ok)), replace=False))
        rec[f"drops_{k}"], rec[f"rand_{k}"], rec[f"size_{k}"] = rd, rnd, msz
        rec[f"gaze_{k}"] = int(lab[d["gp"]])
    res.append(rec)
    if (i + 1) % 20 == 0:
        torch.save(res, OUT)
        print(f"  {i+1}/{N}  ({(time.time()-t0)/60:.1f} min)")

torch.save(res, OUT)
print(f"done in {(time.time()-t0)/60:.1f} min -> {OUT}")

## 4. TEST 1 — super-additivity: were patches redundant?

Uses the cached per-patch drops. The question that decides whether ten notebooks were mis-measured.

In [ ]:
print(f"{'K':>4}{'median ratio':>15}{'mean ratio':>13}{'ratio>1.5':>12}{'ratio<1':>10}")
print("-" * 56)
for k in KS:
    ratios = []
    for r, d in zip(res, data):
        lab = LAB[k][r["idx"]]
        for j in range(k):
            sel = ((lab == j) & (~sinks[:L_v])).nonzero().squeeze(-1)
            if len(sel) < 2:
                continue
            indiv = float(d["drops"][sel].sum())
            if abs(indiv) < 1e-3:
                continue
            ratios.append(float(r[f"drops_{k}"][j]) / indiv)
    a = np.array(ratios)
    print(f"{k:>4}{np.median(a):>15.2f}{a.mean():>13.2f}"
          f"{(a > 1.5).mean():>11.0%}{(a < 1).mean():>10.0%}")

print("\n  ratio >> 1  -> patches inside a region cover for each other;")
print("                 per-patch ablation UNDERSTATED importance and the whole")
print("                 patch-level chain needs redoing at region level")
print("  ratio ~ 1   -> no redundancy; per-patch ablation was measuring the right thing")

## 5. TEST 2 — did grouping help, or just size?

In [ ]:
print(f"{'K':>4}{'region':>10}{'random':>10}{'diff':>9}{'paired p':>11}{'region>rand':>13}")
print("-" * 58)
for k in KS:
    reg = np.concatenate([r[f"drops_{k}"].numpy() for r in res])
    rnd = np.concatenate([r[f"rand_{k}"].numpy() for r in res])
    m = np.isfinite(reg) & np.isfinite(rnd)
    reg, rnd = reg[m], rnd[m]
    p = wilcoxon(reg, rnd)[1]
    print(f"{k:>4}{reg.mean():>10.3f}{rnd.mean():>10.3f}{reg.mean()-rnd.mean():>+9.3f}"
          f"{p:>11.3g}{(reg > rnd).mean():>12.0%}")

k = 8
reg = np.concatenate([r[f"drops_{k}"].numpy() for r in res])
rnd = np.concatenate([r[f"rand_{k}"].numpy() for r in res])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist([reg, rnd], bins=40, label=["contiguous region", "random, same size"])
ax[0].legend(fontsize=8); ax[0].set_xlabel("drop in answer logprob"); ax[0].set_title(f"K={k}")
ax[1].scatter(rnd, reg, s=12, alpha=.5)
lim = [min(rnd.min(), reg.min()), max(rnd.max(), reg.max())]
ax[1].plot(lim, lim, "k--", lw=1)
ax[1].set_xlabel("random same-size"); ax[1].set_ylabel("contiguous region")
ax[1].set_title("above the line = grouping matters")
plt.tight_layout(); plt.show()

## 6. TEST 3 — is the region the answer needed the one you were looking at?

In [ ]:
for k in KS:
    same, rank, outside = 0, [], []
    for r in res:
        rd, gr = r[f"drops_{k}"], r[f"gaze_{k}"]
        same += int(int(rd.argmax()) == gr)
        rank.append(torch.argsort(rd, descending=True).tolist().index(gr) + 1)
        tot = float(rd.abs().sum().clamp_min(1e-9))
        outside.append(1 - float(rd[gr].abs()) / tot)
    print(f"K={k:<3} top-drop region is the gaze region {same/len(res):>5.0%} "
          f"(chance {1/k:.0%})  |  gaze rank {np.mean(rank):.2f}/{k} "
          f"(chance {(k+1)/2:.1f})  |  {np.mean(outside):.0%} of damage outside it")

k = 8
by = defaultdict(list)
for r, d in zip(res, data):
    by[d["type"]].append(int(int(r[f"drops_{k}"].argmax()) != r[f"gaze_{k}"]))
print(f"\nK={k}, fraction where the answer needed a DIFFERENT region:")
for t in sorted(by, key=lambda t: -np.mean(by[t])):
    print(f"   {t:<38} {np.mean(by[t]):.0%}")

## 7. Verdict

**Test 1 is the one that matters.**

* **ratio >> 1** (say median above ~1.5) -> patches inside an object cover for each other. Per-patch
  ablation understated importance, and every earlier result — the teacher bake-off, the FRM training,
  the whole 22% vs 26% band — was measured on the wrong unit and must be redone at region level.
* **ratio ~ 1** -> no redundancy. Per-patch ablation was fine, the weak signal is real, and the last
  excuse for FRM is gone.

**Test 2 guards against the trap v1 fell into.** If contiguous regions do no better than random
same-size sets, then "regions" are not a meaningful unit here whatever the clustering looks like, and
Test 1's ratio is about mask size rather than about objects.

**Test 3 is the FRM premise**, now at a granularity where the question is well posed. Note the chance
floors move with K: at K=8 the gaze region is top 12.5% of the time by luck, and its expected rank is
4.5.

If Test 1 says redundancy is real, the next build is the object-level FRM — pooled region embedding
as query, ~8 regions as keys, predicting 8 numbers instead of 63. If Test 1 says additive, then the
patch-level numbers stand and Stage 2b should be dropped for the eccentricity baseline.